Eren burada grafikler çizip veriyi tanıyacak.

# **Data Merge**

In [ ]:
import pandas as pd

# Dosyaları okuyoruz.

df_customers = pd.read_csv('../data/processed/clean_customers.csv')
df_orders = pd.read_csv('../data/processed/clean_orders.csv')
df_details = pd.read_csv('../data/processed/clean_ordersdetails.csv')
df_products = pd.read_csv('../data/processed/clean_products.csv')
df_categories = pd.read_csv('../data/processed/clean_categories.csv')

print('Dosyalar başarıyla okundu.')

In [ ]:
# Detaylar ve ürünler tablolarını birleştiriyoruz.
merge1 = df_details.merge(df_orders, on='OrderID', how='left')

# Birleştirilmiş tabloyu ürünler tablosu ile birleştiriyoruz.
merge2 = merge1.merge(df_products, on='ProductID', how='left')

# Birleştirilmiş tabloyu kategoriler tablosu ile birleştiriyoruz.
merge3 = merge2.merge(df_categories, on='CategoryID', how='left')

# Birleştirilmiş tabloyu müşteriler tablosu ile birleştiriyoruz.   
df_merged = merge3.merge(df_customers, on='CustomerID', how='left')

#Order_Date sütununu datetime formatına çeviriyoruz.
df_merged['Order_Date'] = pd.to_datetime(df_merged['OrderDate'])

df_merged.head()

In [ ]:
print(f"Toplam Satır Sayısı: {len(df_merged)}")
print(df_merged.isnull().sum()) 

Burada 73 satırda eksik bilgi tespit edildi. Analiz ve grafikler sırasında problem çıkarmaması adına bu 73 satırı veriden çıkartmayı tercih ediyorum. Toplam verinin %3.4ünü oluşturuyor bu kayıp.

In [ ]:
# 1. Adım: Temizliği yapıp yeni bir değişkene atıyoruz (df_master)
# subset=['OrderDate'] -> "Sadece Tarih sütunu boşsa sil" demek.
df_master = df_merged.dropna(subset=['OrderDate'])

#Hatalı verileri bir tabloda toplayalım.
df_errors = df_merged[df_merged['OrderDate'].isnull()]

# 2. Adım: Kontrol
print(f"Eski Satır Sayısı: {len(df_merged)}")
print(f"Yeni Satır Sayısı: {len(df_master)}")
print(f"Kenara ayrılan hatalı kayıt sayısı: {len(df_errors)}")
print("Kalan Boş Tarih Sayısı:", df_master['OrderDate'].isnull().sum())


# **Feature Engineering Ve Analiz**

In [ ]:
#Verinin içinden ayları,günleri çekiyoruz.
df_master['Order_Month'] = df_master['Order_Date'].dt.month
df_master['Order_Day'] = df_master['Order_Date'].dt.day

#Kontrol
print(df_master[['Order_Date', 'Order_Month', 'Order_Day']].head())

Resampling

In [ ]:
import matplotlib.pyplot as plt

df_master['OrderDate'] = pd.to_datetime(df_master['OrderDate'])

# 1. Önce tarihi 'Index' (Kitap fihristi) yapıyoruz ki Pandas zamanı anlasın.
# (Geçici bir kopya üzerinde çalışıyoruz, orijinal df_master bozulmasın)
df_trend = df_master.set_index('OrderDate')

# 2. 'M' = Month (Ay). Yani veriyi aylık olarak sıkıştır ve topla.
monthly_sales = df_trend['TotalAmount'].resample('ME').sum()

# 3. Grafiği Çiz (Basit bir çizgi grafik)
plt.figure(figsize=(12, 6)) # Tablonun boyutu (Genişlik, Yükseklik)
monthly_sales.plot(color='#2ecc71', linewidth=3, marker='o') # Yeşil renk, kalın çizgi

plt.title('Aylık Ciro Trendi (Zaman İçindeki Değişim)', fontsize=16)
plt.xlabel('Tarih', fontsize=12)
plt.ylabel('Toplam Satış ($)', fontsize=12)
plt.grid(True, alpha=0.3) # Arkaya hafif kareli kağıt deseni
plt.show()

In [ ]:
print("Verideki En Son Sipariş Tarihi:", df_master['OrderDate'].max())

Yukarıda verimizi aylara bölerek, o aylardaki toplam ciroları toplayıp mevimsellik analizi uyguladım. Ve nisan ayındaki (yani verinin son ayındaki) anormal düşüş dikkatimi çekti.
Verilerin hangi günde bittiğini kontrol ettim ve burda gördüğümüz üzere diğer aylarda 30'ar günlük veriler toplanırken, Nisan ayındaki son veri girişi 5 Nisan'da yapılmış. Ve böyle bir yanılgıya sebep olmuş.

In [ ]:
#Yukarıdaki yanılgıyı gidermek ve daha gerçekçi bir trend görmek için son yarım ayı temizleyelim.
monthly_sales_clean = monthly_sales.iloc[:-1] 

# Şimdi temiz grafiği çizelim
plt.figure(figsize=(14, 7))
monthly_sales_clean.plot(color='#27ae60', linewidth=3, marker='o') # Renk değiştirdim (Yeşil)

plt.title('Aylık Ciro Trendi (Yarım Aylar Temizlendi)', fontsize=16)
plt.xlabel('Tarih')
plt.ylabel('Ciro ($)')
plt.grid(True, alpha=0.3)
plt.show()

**Kategori Bazlı Performans İncelemesi (Pareto)** 

In [ ]:
import seaborn as sns

# 1. Kategorilere göre Ciro Toplamı
# reset_index() diyerek tabloyu düzeltiyoruz ki grafik çizebilelim
category_performance = df_master.groupby('CategoryName')['TotalAmount'].sum().reset_index()

# 2. Çoktan aza sırala (Şampiyon en üstte olsun)
category_performance = category_performance.sort_values('TotalAmount', ascending=False)

# 3. Yüzdelik Katkıyı Hesapla (Pareto Mantığı)
# Her kategori toplam cironun % kaçı?
total_revenue = category_performance['TotalAmount'].sum()
category_performance['Share (%)'] = (category_performance['TotalAmount'] / total_revenue) * 100

# Tabloyu görelim
print(category_performance)

# --- Görselleştirme ---
plt.figure(figsize=(12, 6))
sns.barplot(data=category_performance, x='TotalAmount', y='CategoryName', palette='viridis')

plt.title('Hangi Kategori Şirketi Sırtlıyor?', fontsize=16)
plt.xlabel('Toplam Ciro ($)')
plt.ylabel('Kategori')
plt.grid(axis='x', alpha=0.3)
plt.show()

Buradan elde edilen çıktılar:

1- Beverages ve Dairy Products Toplam Cironun %40ını oluşturuyor. Bu tarafta bir sorun yaşanmayacağından emin olunmalı.

2-Produce ve Grains/Cereals kategorileriyse en alt tabakayı oluşturuyor. Burada kritik bir soru olarak:
''Acaba bu ürünlerin kar marjı mı düşük, yoksa müşteriler bizi manav olarak görmüyor mu?'' araştırılabilir.

# **Öneri Motoru İçin Matris Hazırlığı**


In [ ]:
# Amaç: Hangi müşteri (Satır), Hangi üründen (Sütun) kaç tane almış?

# 1. Pivot Table
user_item_matrix = df_master.pivot_table(
    index='CustomerID',      # Satırlar: Müşteriler
    columns='ProductName',   # Sütunlar: Ürünler (İstersen ProductID de yapabilirsin ama isimler daha okunaklı)
    values='Quantity',       # Değerler: Kaç tane aldığı
    aggfunc='sum',           # Aynı ürünü birden fazla kez aldıysa topla
    fill_value=0             # Almadığı ürünlere 0 yaz (NaN kalmasın)
)

# 2. Matrisi Kontrol Et 
print("Matris Boyutu:", user_item_matrix.shape) # (Müşteri Sayısı x Ürün Sayısı) olmalı
user_item_matrix.head(10)

#user_item_matrix.to_csv('../data/processed/user_item_matrix.csv')
#print("'user_item_matrix.csv' de oluşturuldu.")

In [ ]:
import numpy as np

# 1. Sparsity (Seyreklik) Hesabı
# Matrisin yüzde kaçı 0?
toplam_hucre = user_item_matrix.size # 89 x 77
dolu_hucre = np.count_nonzero(user_item_matrix)
bosluk_orani = (1 - (dolu_hucre / toplam_hucre)) * 100

print(f"Matris Boyutu: {user_item_matrix.shape}")
print(f"Matrisin Doluluk Oranı: %{100 - bosluk_orani:.2f}")
print(f"Matrisin Seyreklik (Sparsity) Oranı: %{bosluk_orani:.2f}")

# YORUM:
if bosluk_orani > 99:
    print("UYARI: Veri çok seyrek! 'Matrix Factorization' veya 'ALS' kullanılması önerilir.")
else:
    print("BİLGİ: Veri yoğunluğu makul. 'User-Based Collaborative Filtering' çalışabilir.")

# 2. Alternatif Versiyon: Binary Matrix (0 ve 1)
user_item_matrix_binary = user_item_matrix.applymap(lambda x: 1 if x > 0 else 0)

user_item_matrix_binary.head(10)
# Bunu da yedekleyelim
#user_item_matrix_binary.to_csv('../data/processed/recommendation_matrix_binary.csv')
#print("'recommendation_matrix_binary.csv' de oluşturuldu.")

| Dosya Adı | Açıklama | İçerik Tipi |
| :--- | :--- | :--- |
| **`user_item_matrix.csv`** | Orijinal Etkileşim Matrisi | Satın alınan **Adet (Quantity)** bilgisi içerir. |
| **`recommendation_matrix_binary.csv`** | Binary Matris | Sadece **var/yok (1 veya 0)** bilgisi içerir. Implicit feedback modelleri için uygundur. |

# **RFM Analizi**

In [ ]:
# Verideki en son alışveriş tarihini bulalım
son_tarih = df_master['OrderDate'].max()

# Analiz Tarihi (Bugün) olarak, son tarihin üzerine 2 gün ekleyelim.
# Neden? Son gün alışveriş yapanın Recency değeri 0 çıkmasın, en azından 2 gün çıksın diye.
analiz_tarihi = son_tarih + pd.Timedelta(days=2)

print(f"Verideki Son Tarih: {son_tarih}")
print(f"Bizim Analiz Tarihimiz (Bugün kabul ettiğimiz): {analiz_tarihi}")

In [ ]:
# Müşteri bazında gruplama yapıyoruz
rfm = df_master.groupby('CustomerID').agg({
    # RECENCY (Yenilik) Hesabı:
    # (Analiz Tarihi) - (Müşterinin Son Alışveriş Tarihi) = Kaç gün geçmiş?
    'OrderDate': lambda date: (analiz_tarihi - date.max()).days,

    # FREQUENCY (Sıklık) Hesabı:
    # Kaç tane 'EŞSİZ' sipariş numarası (OrderID) var?
    'OrderID': 'nunique',

    # MONETARY (Parasal) Hesabı:
    # Toplam ne kadar para harcadı?
    'TotalAmount': 'sum'
})

# Sütun isimleri karışık gelmesin diye düzeltelim
rfm.columns = ['Recency', 'Frequency', 'Monetary']

# Sonucu görelim
rfm.head()



In [ ]:
print(rfm.describe().T) # .T transpoze eder, daha rahat okunur

In [ ]:
# 1. Puanlama (1-5 arası not veriyoruz)
# qcut: Veriyi eşit parçalara böler (Çeyreklikler)

# Recency: Düşük olan (yakın tarih) 5 puan alır. O yüzden etiketler [5, 4, 3, 2, 1]
rfm["RecencyScore"] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])

# Frequency: Yüksek olan 5 puan alır. (rank(method='first') çakışmaları önlemek için)
rfm["FrequencyScore"] = pd.qcut(rfm['Frequency'].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])

# Monetary: Yüksek olan 5 puan alır.
rfm["MonetaryScore"] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

# 2. Skorları Birleştirip "RFM SKORU" oluşturma
# Örn: 555 (Şampiyon), 111 (Kaybedilmiş)
rfm["RFM_SCORE"] = (rfm['RecencyScore'].astype(str) + 
                    rfm['FrequencyScore'].astype(str))

# 3. SEGMENT HARİTASI (En Zevkli Kısım) 🗺️
# Hangi puan ne anlama geliyor?
seg_map = {
    r'[1-2][1-2]': 'Hibernating (Uykudakiler)',
    r'[1-2][3-4]': 'At Risk (Riskli Grup)',
    r'[1-2]5': 'Can\'t Loose (Kaybedemeyiz)',
    r'3[1-2]': 'About to Sleep (Uyumak Üzere)',
    r'33': 'Need Attention (Dikkat İster)',
    r'[3-4][4-5]': 'Loyal Customers (Sadık Müşteriler)',
    r'41': 'Promising (Umut Vaat Eden)',
    r'51': 'New Customers (Yeni Gelenler)',
    r'[4-5][2-3]': 'Potential Loyalists (Potansiyel Sadıklar)',
    r'5[4-5]': 'Champions (Şampiyonlar)'
}

# 4. Haritayı Uygula
rfm['Segment'] = rfm['RFM_SCORE'].replace(seg_map, regex=True)

# Sonucu Görelim: Kim hangi gruba düştü?
print(rfm[['Segment', 'Recency', 'Frequency', 'Monetary']].head(10))


In [ ]:
# Segmentlere göre ortalamaları alalım
segment_analizi = rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'RFM_SCORE': 'count' # Bu bize kişi sayısını verir
})

# Sütun adını düzeltelim
segment_analizi.rename(columns={'RFM_SCORE': 'Kişi Sayısı'}, inplace=True)

print(segment_analizi.sort_values('Monetary', ascending=False))

In [ ]:
# Segmentlere göre ortalamaları alalım
segment_analizi = rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'RFM_SCORE': 'count' # Bu bize kişi sayısını verir
})

# Sütun adını düzeltelim
segment_analizi.rename(columns={'RFM_SCORE': 'Kişi Sayısı'}, inplace=True)

print(segment_analizi.sort_values('Monetary', ascending=False))

In [ ]:
rfm.reset_index().to_csv('../data/processed/rfm_segments.csv', index=False)

# **COHESION ANALYSIS**

In [ ]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import calculate_monthly_growth, calculate_cohort_matrix

# 2. Growth Testi
print("📈 Büyüme Oranları:")
growth = calculate_monthly_growth(df)
print(growth.tail(5)) # Son 5 ayı görelim

# 3. Cohort Testi (Retention)
print("\nCohort Retention Matrisi (İlk 5 Satır):")
retention_matrix = calculate_cohort_matrix(df)

# Görselleştirmeden önce matrisin kendisine bakalım
# 1. sütun her zaman 1.0 (yani %100) olmalı.
print(retention_matrix.iloc[:5, :5])